# NumPy Practice Workbook — Smart Factory Sensor Analytics

A two-phase, hands-on NumPy workbook built around a synthetic industrial sensor dataset. Every task is solved by you first; hints and full solutions are available on request through an imported helper, not printed inline.

## Learning Objectives

By the end of this workbook you should be able to:

- Create, inspect, and reshape NumPy arrays confidently, in 1D, 2D, and 3D
- Index, slice, mask, and fancy-index arrays to extract exactly the data you need
- Apply broadcasting correctly, including across non-trailing axes
- Compute aggregate, axis-based, and NaN-aware statistics
- Recognize the view-vs-copy distinction and avoid the bugs it causes
- Clean realistic, imperfect array data (missing values, sensor faults, anomalies)
- Choose vectorized NumPy operations over Python loops, and know when a loop is still fine
- Recognize which NumPy tool fits a new, unstated problem — not just recall syntax

## Workbook Roadmap

**Phase 1 — Learning.** Work through the building blocks of NumPy one at a time. Tasks may name the function or concept being practiced. Use `hint(n)` if stuck, `answer(n)` once you've made a real attempt.

**Phase 2 — Testing.** Independent problems that generally do *not* name the function you need — you decide the right approach. Difficulty increases from easy, single-concept checks through to debugging, decision-making, and interview-style reasoning questions.

No theory is provided beyond what's needed to attempt each task — this is a practice workbook, not a textbook.

## Dataset — Smart Factory Sensor Network (synthetic)

**Domain.** A synthetic manufacturing facility with 12 machines, each instrumented with three sensors — temperature (°C), vibration (mm/s), and pressure (kPa) — sampled hourly over 5 days (120 hourly readings per machine). Production output and downtime are also tracked.

**Why synthetic.** The workbook needs realistic imperfections (missing readings, sensor spikes, a wiring fault) placed at *known* locations so tasks are checkable, while staying purely numeric — the right level of complexity for a NumPy-focused workbook rather than a pandas/tabular one. The generation is fully reproducible via a fixed seed.

**Arrays produced:**

| Array | Shape | dtype | Description |
|---|---|---|---|
| `temperature` | (12, 120) | float64 | Hourly °C readings. Contains ~15 fault spikes and ~10 missing (NaN) readings. |
| `vibration` | (12, 120) | float64 | Hourly mm/s readings. Contains ~15 fault spikes. |
| `pressure` | (12, 120) | float64 | Hourly kPa readings. Contains 6 zero-value wiring faults. |
| `production` | (12, 5) | int64 | Daily units produced per machine. |
| `downtime` | (12,) | float64 | Total downtime hours per machine over the period. |
| `machine_ids` | list[str] | — | Labels `M01`...`M12`, for reference only. |

**Data-quality issues (intentional):** missing values, sensor fault spikes, a physically impossible zero-pressure fault, and machine-to-machine baseline variation — all realistic for industrial telemetry, and all used deliberately across the workbook's tasks.

## Environment / Setup

In [ ]:
import numpy as np

### Load the Dataset

Run this cell once at the start of your session. It deterministically generates the sensor arrays described above.

In [ ]:
from practice_system import load_dataset

temperature, vibration, pressure, production, downtime, machine_ids, n_machines, n_hours = load_dataset()

print("temperature:", temperature.shape, "| vibration:", vibration.shape,
      "| pressure:", pressure.shape)
print("production:", production.shape, "| downtime:", downtime.shape)

### Interactive System Import

Run this once to enable `hint(n)` and `answer(n)` for every task in this workbook.

In [ ]:
from practice_system import hint, answer

---
# PHASE 1 — LEARNING

Work through these tasks in order. Each introduces a building block you'll need for Phase 2. Use `hint(n)` and `answer(n)` freely — this phase is about learning, not testing yourself.

### Task 1 — Array Creation from Python Lists

Create a 1D NumPy array from the Python list `[65.2, 66.8, 64.1, 70.5, 68.9]` representing five temperature readings from Machine M01. Then create a 2D NumPy array from the nested list `[[65.2, 66.8], [70.1, 69.4], [63.5, 64.0]]` representing readings from 3 machines across 2 hours. Print both arrays along with their types.

### Task 2 — Creation with zeros / ones / full

You need placeholder arrays to initialize tracking structures before analysis. Create: (a) a 1D array of 12 zeros representing an 'alerts triggered' counter for each machine, (b) a 2D array of shape (12, 5) filled with ones representing a 'machine active' flag for each of the 5 days, and (c) a 1D array of 12 elements all set to the value 100, representing a default 'health score' for each machine.

### Task 3 — arange & linspace

Create an array of hour markers from 0 to 119 (one for every hourly reading) using a step-based method, and a separate array of 6 evenly spaced percentile thresholds between 0 and 1 (inclusive) for later use in reporting.

### Task 4 — Identity Matrix with eye

Create a 5x5 identity matrix that will be used later as a baseline 'no cross-sensor interference' calibration matrix.

### Task 5 — Reproducible Random Arrays

Using a fixed seed of 7 for reproducibility, generate: (a) a 3x3 array of random floats between 0 and 1, (b) a 1D array of 10 random integers between 1 and 100 (lower bound inclusive, upper bound exclusive), and (c) a 4x4 array of values drawn from a standard normal distribution. Re-seed before each generation so every part is independently reproducible.

### Task 6 — Array Attributes

Using the `temperature` sensor array, print its shape, number of dimensions, total element count, and data type.

### Task 7 — Data Type Casting (astype)

Create a float32 copy of the `production` array for a downstream calculation requiring decimal precision, without modifying the original. Then downcast the `pressure` array (currently float64) to float32 and compare memory usage in bytes before and after using `.nbytes`.

### Task 8 — 1D Indexing

Using the `downtime` array, retrieve the downtime value of the first machine and the downtime value of the last machine, without hardcoding the array length.

### Task 9 — 2D Indexing

From the `temperature` array, retrieve the full row of readings for machine index 3 (M04), and separately retrieve the single reading for machine index 3 at hour index 50.

### Task 10 — 1D Slicing

Using the `downtime` array, extract the downtime values for the first 5 machines, extract every second machine's downtime starting from index 1, and produce a reversed copy of the full array.

### Task 11 — 2D Slicing

From the `temperature` array, extract readings for the first 4 machines across the first 24 hours (day 1) only.

### Task 12 — Boolean Masking

Identify which temperature readings exceed 90°C (a clear anomaly threshold) and extract only those readings into a flat 1D array. Also report how many such readings exist.

### Task 13 — Fancy Indexing

Using the `downtime` array, retrieve the downtime for machines at indices 0, 5, and 9 in a single operation. Then, from the `temperature` array, retrieve the specific readings at (machine 2, hour 10), (machine 7, hour 50), and (machine 11, hour 100) — three individual cells, not a submatrix.

### Task 14 — Locating & Replacing with np.where

Using the `vibration` array, find the (machine, hour) index locations where vibration exceeds 4.5 mm/s (a fault threshold). Then create a cleaned copy of `vibration` where any value above 4.5 is capped to exactly 4.5, leaving all other values unchanged.

### Task 15 — Reshaping Arrays

The `production` array has shape (12, 5). Reshape it into a 1D array of 60 values, then reshape that back into shape (5, 12) so each row represents a day instead of a machine. Verify the total element count never changes.

### Task 16 — Flatten vs Ravel

Make a working copy of the `temperature` array called `temp_sandbox` (using `.copy()`, so the shared dataset isn't disturbed by this demo). Create `flat_copy = temp_sandbox.flatten()`, change its first element, and check whether `temp_sandbox` was affected. Separately, create `flat_view = temp_sandbox.ravel()`, change its first element too, and check whether `temp_sandbox` was affected this time.

### Task 17 — View vs Copy

Make a working copy of the `pressure` array called `pressure_demo` (using `.copy()`, so the shared dataset isn't disturbed by this demo). Slice its first 3 rows into a variable called `subset`, change one value inside `subset`, and check whether `pressure_demo` itself changed as a result. Then make a second fresh copy called `pressure_demo2`, slice its first 3 rows into a variable called `subset2`, but this time call `.copy()` on that slice too before changing a value in it — and check whether `pressure_demo2` changed this time.

### Task 18 — Elementwise Arithmetic

Compute a combined 'stress index' array using the formula `stress_index = (vibration * 10) + (pressure - 100)`, applied elementwise to the two full arrays directly — no loop needed.

### Task 19 — Broadcasting (scalar & 1D-2D)

Convert the entire `temperature` array from Celsius to Fahrenheit using F = C*9/5+32, applied to the whole 2D array in one expression. Then create a 1D array `daily_targets` containing 5 target production numbers of your own choosing (matching the 5 days in `production`), and add it to every row of the `production` array in a single operation.

### Task 20 — Broadcasting Across Rows (Calibration)

Create a 1D array `calibration_offsets` of length 12, containing any reasonable offset value for each machine (for example, random floats between -1 and 1). Apply this offset to every hourly reading in the `temperature` array so that offset[i] is added to every value in row i — without using a loop.

### Task 21 — Aggregate Statistics

Compute the overall mean, minimum, maximum, and standard deviation of the entire `pressure` array, across all machines and hours combined.

### Task 22 — Axis-based Aggregation

Compute the average production per machine (across all 5 days) and separately the average production per day (across all 12 machines). Identify which machine has the highest average production.

### Task 23 — Cumulative Functions

Using the `production` array, compute for each machine the running cumulative total production across the 5 days, and identify the index of the day (0-4) at which each machine's cumulative total first reaches or exceeds 600 units.

### Task 24 — Clipping Values (np.clip)

Sensor specs say vibration readings should never physically exceed 5.0 mm/s regardless of what was recorded. Produce a cleaned vibration array where every value is capped to a valid range of 0 to 5.0 mm/s in a single operation.

### Task 25 — Unique Values & Counts

Round the `pressure` array to the nearest whole kPa and determine how many distinct rounded pressure values occur across the dataset, along with how many times each occurs.

### Task 26 — any / all for Validation Checks

Write two checks: (a) confirm whether *any* machine has zero downtime at all, and (b) confirm whether *all* machines produced at least 150 units on every single day.

### Task 27 — Handling Missing Data

The `temperature` array contains missing readings recorded as NaN due to sensor dropout. Count how many NaN values exist in total, compute the mean temperature ignoring those missing values, and create a cleaned array where every NaN is replaced with that overall mean.

### Task 28 — Sorting

Sort the `downtime` array from lowest to highest and separately produce a version sorted from highest to lowest.

### Task 29 — Argsort & Ranking

Determine which 3 machines have the highest total downtime, returning their machine indices in order from worst to best (not the downtime values themselves).

### Task 30 — Axis-based Sorting (2D)

Sort each machine's 5-day production figures independently (each row sorted internally), without disturbing which machine each row belongs to. Then separately sort each day's column of production values across machines.

### Task 31 — Concatenation

Create a new array of shape (12, 2) with any reasonable values, representing two additional days of production data. Combine it with the existing `production` array so the result has shape (12, 7), with the new days appended after the existing 5.

### Task 32 — Vertical / Horizontal Stacking

Create a 1D array of length 5 with any reasonable values, representing 5 days of production data for a new machine (machine 13) that has just come online. Add it as a new row to the `production` array. Separately, create another array of shape (12, 5) with any reasonable values, and demonstrate combining it with `production` side-by-side into a (12, 10) array.

### Task 33 — Splitting Arrays

Split the `production` array (12 machines) into 3 equal groups of 4 machines each, representing 3 factory zones, without manually slicing each group by hand.

### Task 34 — Tile & Repeat

Create a 1D array representing a repeating 3-day maintenance shift pattern `[1,2,3]`, tiled out to cover 12 machines (the whole pattern repeated end-to-end). Separately, create an array where each shift number is repeated 4 times in a row instead (element-wise repetition), also covering 12 machines.

### Task 35 — Linear Algebra Basics

Define a 1D array `energy_weights` of length 5, containing any reasonable energy-weight value for each of the 5 days. Compute the total weighted energy consumption per machine using matrix multiplication between the `production` array and `energy_weights`.

### Task 36 — Multi-dimensional Arrays (3D Stacking)

Combine the `temperature`, `vibration`, and `pressure` arrays into a single 3D array of shape (3, 12, 120) representing (sensor_type, machine, hour). Then compute the mean reading for each sensor type across all machines and hours, and separately the mean for each machine across all sensor types and hours.

### Task 37 — Vectorization vs Python Loops

Compute the elementwise squared difference between `temperature` and 65 (a baseline) two ways: once using a plain Python nested for-loop over every element, and once using a fully vectorized NumPy expression. Time both approaches and compare.

### Task 38 — Saving & Loading Arrays

Save the `production` array to disk in NumPy's native binary format, then load it back and confirm the loaded array is identical to the original. Also save `temperature`, `vibration`, and `pressure` together into a single compressed archive file, then load and access each array back by name.

---
# PHASE 2 — TESTING

These problems generally do not tell you which function or approach to use — that's the
point. Work through them in order; difficulty rises from easy through interview-level
reasoning, with some mixing near the end so you can't rely on a predictable pattern.

### Task 39 — Rebuild the Reading

A colleague sends you a single day's worth of hourly vibration readings for one machine as a plain Python list of 24 floats (invent any 24 reasonable values yourself). Convert it into a structure that supports elementwise math and array-based aggregation, and report how many readings it contains without using `len()`.

### Task 40 — Describe the Array

Given the `pressure` array, report in one line each: how many machines it covers, how many hourly readings per machine, and what numeric precision the values are stored in.

### Task 41 — Grab the Weekend

Assuming hour index 96 marks the start of the final full day (day 5) in the 120-hour window, extract every machine's readings for just that final day from the `temperature` array.

### Task 42 — Flag the Risk

Identify every individual pressure reading in the dataset that falls below 95 kPa or above 108 kPa, and return them as a flat array of the offending values.

### Task 43 — One Number to Rule Them All

Report the single highest vibration reading recorded anywhere in the dataset, across every machine and every hour.

### Task 44 — Same Data, New Shape

The `production` array (12 machines x 5 days) needs to be restructured into a format with 3 rows instead, with the total value count staying identical. Determine an appropriate new shape and perform the conversion, letting NumPy infer one of the dimensions for you.

### Task 45 — Line Them Up

Order the 12 machines from least total downtime to most, and report the values in that order (not the machine identities).

### Task 46 — Worst Hour of the Week

For each machine, determine the single hour (0-119) at which its temperature reading was highest — accounting for the fact that some readings are missing.

### Task 47 — Fair Comparison

Machines run at different baseline temperatures due to their location on the factory floor. Using each machine's own average temperature as its baseline (accounting for missing readings), compute how far above or below that baseline every individual hourly reading is, for the entire array in one operation.

### Task 48 — Trustworthy Average

Your manager asks for the average temperature across the whole dataset. A teammate's script reports `nan`. Diagnose why, and produce a corrected calculation that yields a real number.

### Task 49 — Zone Performance

Machines 0-3 form Zone A, 4-7 form Zone B, and 8-11 form Zone C. Determine which zone had the highest average production per machine across the full 5-day period, and briefly state how you'd verify your answer is trustworthy.

### Task 50 — Cherry-Pick the Flags

You've been given a list of (machine, hour) coordinate pairs flagged for manual review: `[(1,15), (4,88), (9,40), (11,5)]`. Retrieve the exact vibration readings at these specific coordinates in one operation, not one at a time.

### Task 51 — Sensor Sanity Check

Pressure sensors occasionally report exactly 0 due to a wiring fault, which is physically impossible for this equipment. Replace every such faulty reading with the machine's own average pressure (computed from its valid readings only), leaving every other reading untouched.

### Task 52 — Merge the Batches

Simulate a scenario where production tracking was temporarily split across two separate systems: slice the existing `production` array into one part covering machines 0-5 and another covering machines 6-11 (each still with the original 5 days), then combine them back into a single (12,5) array in the original machine order. Separately, slice `production` into two parts of shape (12,3) and (12,2), representing days that were logged separately, and combine those back into the full (12,5) array.

### Task 53 — Find the Bug — Broadcasting

A teammate wrote this code to add a per-day bonus target to the production array, but it throws a ValueError:

```python
daily_bonus = np.array([5, 5, 10, 10, 15, 15])
production_with_bonus = production + daily_bonus
```

Identify what's wrong and fix it.

### Task 54 — Find the Bug — Wrong Axis

A teammate wants the average temperature *per hour* (averaged across all 12 machines, producing 120 values) but their code below returns only 12 values instead:

```python
hourly_avg = np.nanmean(temperature, axis=1)
```

Explain what this code actually computed, and correct it to get the intended result.

### Task 55 — Build the Anomaly Report

Produce a short anomaly report: for each machine, determine whether it has ANY vibration reading above 4.5 mm/s, AND report the total count of such anomalous readings across the whole fleet, AND identify the single machine with the most anomalies. Do this without any explicit Python for-loops.

### Task 56 — Find the Bug — Silent Mutation

This function is supposed to return a normalized copy of temperature readings without touching the original dataset, but after calling it once, later calls give wrong results because the original `temperature` array has been altered:

```python
def normalize_first_row(data):
    row = data[0]
    row[np.isnan(row)] = 0
    return (row - row.mean()) / row.std()
```

Explain the bug and rewrite the function so the original array is never modified.

### Task 57 — Speed It Up

This function counts, per machine, how many hourly vibration readings exceed a given threshold, but it's too slow for a dataset with millions of readings:

```python
def count_exceedances(data, threshold):
    counts = []
    for row in data:
        count = 0
        for value in row:
            if value > threshold:
                count += 1
        counts.append(count)
    return counts
```

Rewrite it as a single vectorized expression with no explicit loops, and explain why your version scales better.

### Task 58 — Pick the Right Tool

You need to know, for every hour, which single machine recorded the highest pressure reading. One teammate suggests sorting the pressure array along the machine axis and taking the last row; another suggests a function that directly returns the position of the maximum along that axis. Which approach is more appropriate here, and why? Implement your chosen approach.

### Task 59 — Empty Handed

Write a filter that extracts every temperature reading above 150°C (an impossible sensor fault threshold, well beyond anything in this dataset). Explain what NumPy returns when no elements satisfy a boolean condition, and how you'd safely check for that case before, say, calling `.mean()` on the result.

### Task 60 — What Does This Ranking Mean

A teammate ran `np.argsort(downtime)` and got back `[3, 7, 0, ...]`. They conclude 'Machine 3 has the highest downtime.' Are they interpreting this correctly? Explain what the output actually represents and state the correct conclusion.

### Task 61 — Slice Through the Cube

Using a 3D sensor cube of shape (3, 12, 120) — (sensor_type, machine, hour), with sensor order [temperature, vibration, pressure] — extract just the vibration readings for machine 5 across all hours as a 1D array. (If you still have the `sensor_cube` from Task 36, reuse it; otherwise rebuild it with `np.stack([temperature, vibration, pressure], axis=0)`.)

### Task 62 — Why Did Broadcasting Fail?

Explain, in your own words, the general rule NumPy uses to decide whether two differently-shaped arrays can be combined via broadcasting, and give a concrete example (using shapes, not necessarily this dataset) of two shapes that CANNOT be broadcast together, and why.

### Task 63 — View, Copy, and the Cost of Being Wrong

In a production data pipeline processing millions of sensor readings per hour, explain the practical trade-off between relying on NumPy's default view-returning behavior versus always calling `.copy()` defensively. When would each choice be the right engineering decision? (This is a discussion question — write your answer as a markdown or comment response, then check `answer(63)`.)

### Task 64 — When Is a Loop Actually Fine?

You've now rewritten several loop-based operations as vectorized NumPy expressions for performance. Describe a realistic scenario involving this sensor dataset where a plain Python loop over machines would still be a reasonable engineering choice, despite NumPy's general performance advantage. (Discussion question — write your answer, then check `answer(64)`.)

### Task 65 — flatten, ravel, or reshape(-1)?

Three different techniques can convert a 2D array into 1D: `.flatten()`, `.ravel()`, and `.reshape(-1)`. Explain the meaningful difference between them from a memory-management perspective, and state which you'd choose by default in a memory-constrained production pipeline, and why. (Discussion question — write your answer, then check `answer(65)`.)

### Task 66 — Daily Machine Health Report

Build a single health report for the factory: for each machine, report its average temperature (ignoring missing readings), its total vibration anomaly count using a 4.5 mm/s threshold, its total 5-day production, and its downtime hours. Combine these into one structure per machine (for example, a list of tuples, or a set of aligned arrays), sorted from best overall performer to worst using total production as the ranking criterion. Briefly state what 'best' means here and one limitation of ranking by a single metric.

---
## Final Challenge

### Task 67 — Factory Health Dashboard — Integrated Challenge

Using everything from this workbook, build a complete data-cleaning and reporting pipeline for the sensor dataset:

1. Detect and report the count of missing temperature readings and vibration/pressure anomalies.
2. Produce cleaned versions of all three sensor arrays (NaNs filled with machine-level averages, vibration anomalies capped at a sensible threshold, zero-pressure faults corrected).
3. Compute a per-machine summary combining average readings for each sensor, total production, and downtime.
4. Rank machines by a composite health score of your own design that combines at least three of these factors, briefly justifying your weighting choice.
5. Identify the single machine most in need of maintenance and explain, using the numbers, why.

---
## Completion & Next Steps

If you completed both phases without leaning on `answer()` for most tasks, you have working,
recognition-level command of NumPy's core toolkit — not just the syntax, but when to reach for
each piece.

A few natural next steps:
- Revisit any task where you needed `answer()` and redo it from a blank cell a day later.
- Try applying the view-vs-copy and vectorization judgment from this workbook to a real
  dataset you work with.
- Move on to pandas, where many of these same array-level concepts (boolean masking,
  broadcasting, axis-based aggregation) reappear on top of labeled, tabular data.